# 第4章: 言語解析

問題30から問題35までは、以下の文章`text`（太宰治の『走れメロス』の冒頭部分）に対して、言語解析を実施せよ。問題36から問題39までは、国家を説明した文書群（日本語版ウィキペディア記事から抽出したテキスト群）をコーパスとして、言語解析を実施せよ。

In [76]:
text = """
メロスは激怒した。
必ず、かの邪智暴虐の王を除かなければならぬと決意した。
メロスには政治がわからぬ。
メロスは、村の牧人である。
笛を吹き、羊と遊んで暮して来た。
けれども邪悪に対しては、人一倍に敏感であった。
"""

## 30. 動詞
文章`text`に含まれる動詞をすべて表示せよ。

In [77]:
!pip install janome

In [78]:
from janome.tokenizer import Tokenizer

t = Tokenizer()

# 文章を単語ごとに分割して品詞を確認する
for token in t.tokenize(text):
  res = token.part_of_speech.split(",")

  if (res[0] == "動詞"):
    print(token.surface)

し
除か
なら
し
わから
吹き
遊ん
暮し
来


## 31. 動詞の原型
文章`text`に含まれる動詞と、その原型をすべて表示せよ。

In [79]:
for token in t.tokenize(text):
  res = token.part_of_speech.split(",")

  if (res[0] == "動詞"):
    print(token.base_form)

する
除く
なる
する
わかる
吹く
遊ぶ
暮す
来る


## 32. 「AのB」
文章`text`において、2つの名詞が「の」で連結されている名詞句をすべて抽出せよ。

In [80]:
data = t.tokenize(text)
res = []

for i,token in enumerate(data):
  part_of_speech = token.part_of_speech.split(",")[0]
  res.append([token.surface, part_of_speech])

  if (len(res) >= 3):

    if (res[1][0] == "の"):
      if (res[0][1] == "名詞") and (res[2][1] == "名詞"):
        print(res[0][0])
        print(res[2][0])

    del res[0]

暴虐
王
村
牧人


## 33. 係り受け解析

文章`text`に係り受け解析を適用し、係り元と係り先のトークン（形態素や文節などの単位）をタブ区切り形式ですべて抽出せよ。

In [81]:
!pip install -U ginza ja-ginza

In [82]:
import spacy

# エラーを回避するための設定（config）を定義
config = {
    "components": {
        "compound_splitter": {
            "split_mode": "C",
        }
    }
}

# configを渡してGiNZA（日本語モデル）を読み込む
nlp = spacy.load("ja_ginza", config=config)

In [83]:
# 2. 解析したいテキストを渡す
doc = nlp(text.replace("\n", ""))

# 3. 係り受けの結果を1単語ずつ取り出して表示する
for token in doc:
    # token.text      : その単語（係り元）
    # token.head.text : どこにかかっているか（係り先）
    # token.dep_      : どういう関係か（主語、目的語など）
    print(f"{token.text} -> {token.head.text} ({token.dep_})")

メロス -> 激怒 (nsubj)
は -> メロス (case)
激怒 -> 激怒 (ROOT)
し -> 激怒 (aux)
た -> 激怒 (aux)
。 -> 激怒 (punct)
必ず -> 除か (advmod)
、 -> 必ず (punct)
かの -> 暴虐 (det)
邪智 -> 暴虐 (compound)
暴虐 -> 王 (nmod)
の -> 暴虐 (case)
王 -> 除か (obj)
を -> 王 (case)
除か -> 決意 (advcl)
なけれ -> 除か (aux)
ば -> なけれ (fixed)
なら -> なけれ (fixed)
ぬ -> なけれ (fixed)
と -> 除か (case)
決意 -> 決意 (ROOT)
し -> 決意 (aux)
た -> 決意 (aux)
。 -> 決意 (punct)
メロス -> わから (obl)
に -> メロス (case)
は -> メロス (case)
政治 -> わから (nsubj)
が -> 政治 (case)
わから -> わから (ROOT)
ぬ -> わから (aux)
。 -> わから (punct)
メロス -> 牧人 (nsubj)
は -> メロス (case)
、 -> メロス (punct)
村 -> 牧人 (nmod)
の -> 村 (case)
牧人 -> 牧人 (ROOT)
で -> 牧人 (cop)
ある -> で (fixed)
。 -> 牧人 (punct)
笛 -> 吹き (obj)
を -> 笛 (case)
吹き -> 暮し (advcl)
、 -> 吹き (punct)
羊 -> 遊ん (obl)
と -> 羊 (case)
遊ん -> 暮し (advcl)
で -> 遊ん (mark)
暮し -> 暮し (ROOT)
て -> 暮し (mark)
来 -> て (fixed)
た -> 暮し (aux)
。 -> 暮し (punct)
けれど -> 敏感 (cc)
も -> けれど (fixed)
邪悪 -> 敏感 (obl)
に -> 邪悪 (case)
対し -> に (fixed)
ては -> に (fixed)
、 -> 邪悪 (punct)
人 -> 倍 (compound)
一 -> 倍 (nummod)
倍 ->

## 34. 主述の関係
文章`text`において、「メロス」が主語であるときの述語を抽出せよ。

In [84]:
# 2. 解析したいテキストを渡す
doc = nlp(text.replace("\n", ""))

# 3. 係り受けの結果を1単語ずつ取り出して表示する
for token in doc:
    if (token.dep_ == "nsubj") and (token.text == "メロス"):
      print(token.head.text)

激怒
牧人


## 35. 係り受け木
「メロスは激怒した。」の係り受け木を可視化せよ。

In [87]:
from spacy import displacy

text = "メロスは激怒した。"
doc = nlp(text)

displacy.render(doc,
                style="dep"   # 描画スタイルの指定　dep -> 係り受け
                )

## 36. 単語の出現頻度

問題36から39までは、Wikipediaの記事を以下のフォーマットで書き出したファイル[jawiki-country.json.gz](/data/jawiki-country.json.gz)をコーパスと見なし、統計的な分析を行う。

* 1行に1記事の情報がJSON形式で格納される
* 各行には記事名が"title"キーに、記事本文が"text"キーの辞書オブジェクトに格納され、そのオブジェクトがJSON形式で書き出される
* ファイル全体はgzipで圧縮される

まず、第3章の処理内容を参考に、Wikipedia記事からマークアップを除去し、各記事のテキストを抽出せよ。そして、コーパスにおける単語（形態素）の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

In [90]:
import re
import json

強調文字はそのまま使える

[[記事名#節名|表示文字]]は全部右側だけでOK

[[ファイル:Wikipedia-logo-v2-ja.png|thumb|説明文]]は説明文のみにする

外部リンクは、一律で[]外す

[[Category:ヘルプ|はやみひよう]]は右側だけ

見出しは=消す

In [112]:
import re

text = "Hello [world], this is [python]."

# (.*?) の部分を \1 で受け取り、別の記号【 】で囲み直す
result = re.sub(r'\[(.*?)\]', r'【\1】', text)

print("元のテキスト:", text)
print("処理後テキスト:", result)

元のテキスト: Hello [world], this is [python].
処理後テキスト: Hello 【world】, this is 【python】.


In [114]:
import re

text = "|国章画像 =[[ファイル:Coat_of_arms_of_Egypt.svg|100px|エジプトの国章]]"

# [[ ]] 全体をマッチさせつつ、欲しい部分だけを () で囲む
result = re.sub(r'\[\[.*\|(.*?)\]\]', r'\1', text)

print("処理後:", result)

処理後: |国章画像 =エジプトの国章


In [115]:
text = "|国章画像 =[[ファイル:Coat_of_arms_of_Egypt.svg|100px|エジプトの国章]]"

pattern = r"\[\[ファイル:[^]]*\]\]"
res = re.sub(r"(\[\[ファイル:[^]]*\]\])", r"?\1?", text)
print(res)

|国章画像 =?[[ファイル:Coat_of_arms_of_Egypt.svg|100px|エジプトの国章]]?


In [127]:
def Replace_highlight(text):

  # 強調マークアップの除去
  # 2～5個連続している ' を消す
  res = re.sub(r"'{2,5}", "", text)

  # ファイルの除去
  res = re.sub(r"\[\[\ファイル:([^\]\|])]\]", r"\1", res)

  # {{仮リンク|エジプトの首相|en|Prime Minister of Egypt|label=首相}}
  # 仮リンクの除去
  res = re.sub(r"\{\{仮リンク\|([^\|]*)\|.*\}\}", r"\1", res)


  # MediaWikiマークアップの除去1
  # 例:{{lang|fr|Dieu et mon droit}}
  # {{}}で囲まれた部分を|の一番右側だけにする
  res = re.sub(r"\{\{[^\}]*\|", "", res)

  # 例:
  res = re.sub(r"\{\{|\}\}", "", res)


  #例:[[合同法 (1707年)|1707年合同法]]
  # [[から|までを消す
  # ただし、その間の文字は]以外
  res = re.sub(r"\[\[[^\]]*\|", "", res)

  # "[["と"]]"を消す
  res = re.sub(r"\[\[|\]\]", "", res)


  # MediaWikiマークアップの除去2
  # <ref>や<ref name= ……>などを消す   </ref>はある場合とない場合がある
  # <ref (>以外の0個以上の文字)>(任意の0個以上の文字)0個か1個の</ref>
  res = re.sub(r"<ref[^>]*>.*[</ref>]?", "", res)

  # <br />を消す
  res = re.sub(r"<br ?/?>", "", res)

  return res

In [128]:
documents = []

with open("jawiki-country.json", 'r') as f:
    for line in f:
        line = json.loads(line)   # 辞書型になる keyは"title", "text"

        for word in line["text"].split("\n"):
          print(word)
          print(Replace_highlight(word))

        break

{{otheruses|主に現代のエジプト・アラブ共和国|古代|古代エジプト}}
古代エジプト
{{基礎情報 国
基礎情報 国
|略名 =エジプト
|略名 =エジプト
|漢字書き=埃及
|漢字書き=埃及
|日本語国名 =エジプト・アラブ共和国
|日本語国名 =エジプト・アラブ共和国
|公式国名 ={{lang|ar|'''جمهورية مصر العربية'''}}
|公式国名 =جمهورية مصر العربية
|国旗画像 =Flag of Egypt.svg
|国旗画像 =Flag of Egypt.svg
|国章画像 =[[ファイル:Coat_of_arms_of_Egypt.svg|100px|エジプトの国章]]
|国章画像 =エジプトの国章
|国章リンク =（[[エジプトの国章|国章]]）
|国章リンク =（国章）
|標語 =なし
|標語 =なし
|位置画像 =Egypt (orthographic projection).svg
|位置画像 =Egypt (orthographic projection).svg
|公用語 =[[アラビア語]]
|公用語 =アラビア語
|首都 =[[File:Flag of Cairo.svg|24px]] [[カイロ]]
|首都 =24px カイロ
|最大都市 =カイロ
|最大都市 =カイロ
|元首等肩書 =[[近代エジプトの国家元首の一覧|大統領]]
|元首等肩書 =大統領
|元首等氏名 =[[アブドルファッターフ・アッ＝シーシー]]
|元首等氏名 =アブドルファッターフ・アッ＝シーシー
|首相等肩書 ={{ill2|エジプトの首相|en|Prime Minister of Egypt|label=首相}}
|首相等肩書 =label=首相
|首相等氏名 ={{仮リンク|ムスタファ・マドブーリー|ar|مصطفى مدبولي|en|Moustafa Madbouly}}
|首相等氏名 =ムスタファ・マドブーリー
|面積順位 =29
|面積順位 =29
|面積大きさ =1 E12
|面積大きさ =1 E12
|面積値 =1,010,408
|面積値 =1,010,408
|水面積率 =0.6%
|水面積率 =0.6%
|人口統計年 =2012
|人口統計年 =2012
|人口順位 =
|人口順位 =
|人口

## 37. 名詞の出現頻度
コーパスにおける名詞の出現頻度を求め、出現頻度の高い20語とその出現頻度を表示せよ。

## 38. TF・IDF
日本に関する記事における名詞のTF・IDFスコアを求め、TF・IDFスコア上位20語とそのTF, IDF, TF・IDFを表示せよ。

## 39. Zipfの法則
コーパスにおける単語の出現頻度順位を横軸、その出現頻度を縦軸として、両対数グラフをプロットせよ。